# Data validation: Heavy clay production line

**Goal:** Check that the OEE and downtime data can be trusted before building KPIs and dashboards on them.

**Source:** OEE and downtime datasets from a heavy clay production line (University of Cyprus, Zenodo, 2025), recorded by the Evocon OEE tracking system.
Two periods are provided: before and after a Lean Six Sigma (LSS) programme.

**Output:** `data/processed/oee_shifts.csv` and `data/processed/downtime_events.csv`,
used by the Power BI report

In [1]:
import pandas as pd

from manufacturing_operations_analytics.load import RAW, load_oee, load_downtime

PROCESSED = RAW.parent / "processed"
PROCESSED.mkdir(exist_ok=True)

oee = load_oee()
downtime = load_downtime()

**Structure and coverage**

In [2]:
print("OEE shifts per period:")
print(oee.groupby("period").shift_date.agg(["min", "max", "count", "nunique"]), "\n")
print("Downtime events per period:")
print(downtime.groupby("period").date.agg(["min", "max", "count"]), "\n")

print("OEE nulls:", int(oee.isna().sum().sum()))
nulls = downtime.isna().sum()
print("Downtime nulls:\n", nulls[nulls > 0].to_string(), "\n")
print("Duplicate downtime rows:", downtime.duplicated().sum())
print("Non-positive durations:", (downtime.duration_min <= 0).sum())

OEE shifts per period:
                  min        max  count  nunique
period                                          
after_LSS  2026-01-30 2026-04-03     43       43
before_LSS 2025-09-22 2025-12-18     62       62 

Downtime events per period:
                  min        max  count
period                                 
after_LSS  2026-01-30 2026-04-03    540
before_LSS 2025-09-22 2025-12-18    635 

OEE nulls: 0
Downtime nulls:
 extra_text    25 

Duplicate downtime rows: 0
Non-positive durations: 0


**KPI logic: recomputed vs reported**

Definitions tested:
- Availability = (good + slow production time) / planned time
- Performance = total quantity / ideal quantity
- Quality = good quantity / total quantity
- OEE = availability × performance × quality

In [3]:
calc = pd.DataFrame({
    "availability": (oee.good_prod_time + oee.slow_prod_time) / oee.planned_time,
    "performance": oee.total_qty / oee.ideal_qty,
    "quality": oee.good_qty / oee.total_qty,
})
calc["oee"] = calc.availability * calc.performance * calc.quality

diffs = pd.DataFrame({k: (calc[k] - oee[k]).abs() for k in calc.columns})
print("Max absolute difference, recomputed vs reported:")
print(diffs.groupby(oee.period).max(), "\n")

oee["oee_diff"] = calc.oee - oee.oee
bad = oee.loc[oee.oee_diff.abs() > 0.001,
              ["shift_date", "period", "oee", "oee_diff", "availability", "performance", "quality"]]
print(f"Shifts where reported OEE differs from the product of its components by > 0.001: {len(bad)}")
print(bad.round(4).to_string())

Max absolute difference, recomputed vs reported:
            availability   performance       quality           oee
period                                                            
after_LSS   4.868844e-10  4.931149e-10  4.312556e-10  4.907180e-10
before_LSS  4.916784e-05  4.909462e-05  8.384737e-05  1.185756e-02 

Shifts where reported OEE differs from the product of its components by > 0.001: 10
   shift_date      period     oee  oee_diff  availability  performance  quality
1  2025-09-23  before_LSS  0.5891   -0.0014        0.6236       0.9447   0.9975
2  2025-09-24  before_LSS  0.7498   -0.0036        0.7833       0.9572   0.9951
3  2025-09-25  before_LSS  0.6546   -0.0044        0.7179       0.9118   0.9933
5  2025-09-29  before_LSS  0.6467   -0.0021        0.6717       0.9628   0.9967
15 2025-10-14  before_LSS  0.3790   -0.0015        0.4139       0.9157   0.9958
17 2025-10-16  before_LSS  0.3428   -0.0042        0.3681       0.9314   0.9877
21 2025-10-22  before_LSS  0.2177   -

C:\Users\Athanasia\AppData\Local\Temp\ipykernel_29344\2873136275.py:16: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(bad.round(4).to_string())


**Downtime log vs OEE file**

Stop times in the OEE file are compared with the summed durations in the downtime log, per day. Ratio = log / OEE file, 1.00 means they agree

In [4]:
dt = downtime.assign(stop_class=downtime.stop_type.map({
    "Unplanned": "unplanned",
    "Planned - Included in OEE": "planned_incl",
    "Planned - Not included in OEE": "planned_excl",
}))

daily = dt.groupby(["date", "stop_class"]).duration_min.sum().unstack(fill_value=0) * 60
daily = daily.reindex(columns=["unplanned", "planned_incl", "planned_excl"], fill_value=0)

rec = oee.set_index("shift_date")[["period", "unplanned_stops_time", "planned_stops_time"]].join(daily)
rec["unplanned_ratio"] = rec.unplanned / rec.unplanned_stops_time
rec["planned_all_ratio"] = (rec.planned_incl + rec.planned_excl) / rec.planned_stops_time
rec = rec.replace(float("inf"), float("nan"))

tol = 0.02
off = rec[((rec.unplanned_ratio - 1).abs() > tol) | ((rec.planned_all_ratio - 1).abs() > tol)]
print(f"Days outside ±{tol:.0%}: {len(off)} of {len(rec)}")
print(off.round(2).to_string())

Days outside ±2%: 9 of 105
               period  unplanned_stops_time  planned_stops_time  unplanned  planned_incl  planned_excl  unplanned_ratio  planned_all_ratio
shift_date                                                                                                                                
2026-01-30  after_LSS                6048.0              3178.0     6180.0        3180.0           0.0             1.02               1.00
2026-02-05  after_LSS                2010.0              8610.0     1920.0        2160.0        6480.0             0.96               1.00
2026-02-24  after_LSS                9070.0              2948.0     9000.0        2880.0           0.0             0.99               0.98
2026-02-25  after_LSS                7270.0              1873.0     7320.0        1920.0           0.0             1.01               1.03
2026-03-11  after_LSS               11007.0              1899.0    10980.0        1140.0         720.0             1.00               0.98


**What reduces availability?**

Time not spent running = planned time − production time. If this equals unplanned stops plus planned stops "included in OEE", then planned stops "not included in OEE" are excluded from the availability loss

In [5]:
r = rec.join(oee.set_index("shift_date")[["planned_time", "good_prod_time", "slow_prod_time"]])
r["not_running"] = r.planned_time - r.good_prod_time - r.slow_prod_time
r["leftover"] = r.not_running - r.unplanned_stops_time
r["vs_planned_incl"] = r.leftover - r.planned_incl
r["vs_planned_all"] = r.leftover - (r.planned_incl + r.planned_excl)

print(r.groupby("period")[["leftover", "planned_incl", "planned_excl",
                           "vs_planned_incl", "vs_planned_all"]].median().round(0), "\n")
print("Share of days where leftover matches planned_incl (±60 s):")
print((r.vs_planned_incl.abs() <= 60).groupby(r.period).mean().round(2))

            leftover  planned_incl  planned_excl  vs_planned_incl  \
period                                                              
after_LSS     2276.0        2340.0        1260.0            -11.0   
before_LSS    1330.0        2316.0           0.0           -437.0   

            vs_planned_all  
period                      
after_LSS          -1219.0  
before_LSS         -1463.0   

Share of days where leftover matches planned_incl (±60 s):
period
after_LSS     0.91
before_LSS    0.21
Name: vs_planned_incl, dtype: float64


**Export for Power BI**

Days where the downtime log differs from the OEE file by more than 10% are flagged (`downtime_log_mismatch`), as are shifts whose reported OEE differs from the product of its components (`oee_reported_mismatch`). Nothing is dropped.

In [6]:
bad_days = rec.index[
    ((rec.unplanned_ratio - 1).abs() > 0.10) | ((rec.planned_all_ratio - 1).abs() > 0.10)
]

oee_out = oee.assign(
    oee_reported_mismatch=oee.oee_diff.abs() > 0.001,
    downtime_log_mismatch=oee.shift_date.isin(bad_days),
).drop(columns="oee_diff")

dt_out = dt.assign(downtime_log_mismatch=dt.date.isin(bad_days))

oee_out.to_csv(PROCESSED / "oee_shifts.csv", index=False)
dt_out.to_csv(PROCESSED / "downtime_events.csv", index=False)

print(oee_out.shape, dt_out.shape)
print("Flagged days:", [d.date().isoformat() for d in bad_days])

(105, 16) (1175, 13)
Flagged days: ['2026-04-03']


The KPI components can be reconstructed from the source production fields. The after-LSS OEE values agree almost exactly with the recalculated values. The before-LSS period contains 10 shifts with discrepancies exceeding the selected tolerance, which are retained and flagged for transparency.